In [8]:
%pip install langchain-community langchain-openai unstructured pytesseract pillow tiktoken chromadb

     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ---------------------------------------- 981.5/981.5 kB 6.9 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached psutil-7.2.2-cp37-abi3-win_amd64.whl.metadata (22 kB)
INFO: pip is looking at multiple versions of weasel to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------- ----------- 1.8/2.5 MB 8.6 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 7.2 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ----------------

  You can safely remove it manually.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
%pip install unstructured["local-inference"] pytesseract pillow

   ---------------------------------------- 0.0/538.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/538.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/538.2 kB ? eta -:--:--
   ------------------- -------------------- 262.1/538.2 kB ? eta -:--:--
   ---------------------------------------- 538.2/538.2 kB 1.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.1 MB 1.9 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 4.4 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 4.6 MB/s  0:00:00
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/6.6 MB ? eta -:--:--
   -------------- ------------------------- 2.4/6.6 MB 6.2 MB/s eta 0:00:01
   --------------------------- ------------ 4.5/6.6 MB 7.2 MB/s eta 0:00:01
   ---------------------------------------  6.6/6.6 M


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
%pip install langchain-experimental 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install langchain-google-genai

   ---------------------------------------- 0.0/750.9 kB ? eta -:--:--
   ---------------------------------------- 750.9/750.9 kB 5.6 MB/s  0:00:00

   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   ---------------------------------------- 0/2 [google-genai]
   -----------------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [25]:
import openai
import streamlit as st
import os
import pytesseract
from PIL import Image
from langchain_experimental.text_splitter import SemanticChunker 
from langchain_community.document_loaders import PyPDFLoader, UnstructuredFileLoader, CSVLoader, UnstructuredImageLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

openai.api_key = os.getenv('OPENAI_API_KEY')

In [ ]:
@st.cache_data
def setup_documents(file_path,
                    breakpoint_type='percentile', 
                    breakpoint_threshold=90):
    """
    Xử lý file với Semantic Chunking
    
    Parameters:
    - file_path: đường dẫn file
    - breakpoint_type: 'percentile', 'standard_deviation', 'interquartile'
    - breakpoint_threshold: ngưỡng để xác định điểm cắt
    """
    # 1. Lấy đuôi file (extension) để chọn Loader
    ext = os.path.splitext(file_path)[-1].lower()
    
    try:
        if ext == '.pdf':
            loader = PyPDFLoader(file_path)
            docs_raw = loader.load()
        elif ext == '.txt':
            loader = TextLoader(file_path, encoding='utf-8')
            docs_raw = loader.load()
        elif ext in ['.png', '.jpg', '.jpeg']:
            loader = UnstructuredImageLoader(file_path)
            docs_raw = loader.load()
        else:
            st.error(f"Định dạng file {ext} chưa được hỗ trợ!")
            return []

        # 2. Gom tất cả nội dung văn bản lại
        full_text = "\n".join([doc.page_content for doc in docs_raw])
        
        # 3. TẠO SEMANTIC CHUNKER
        embeddings = OpenAIEmbeddings()
        text_splitter = SemanticChunker(
            embeddings,
            breakpoint_threshold_type=breakpoint_type,
            breakpoint_threshold_amount=breakpoint_threshold
        )
        
        # Tạo danh sách các Document nhỏ
        docs = text_splitter.create_documents([full_text])
        
        return docs

    except Exception as e:
        st.error(f"Lỗi khi xử lý file: {e}")
        return []

2026-03-22 02:45:36.349 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [21]:
def custom_summary(docs,llm, custom_prompt, chain_type, num_summaries):
    custom_prompt = custom_prompt + """:\n\n {text}"""
    COMBINE_PROMPT = PromptTemplate(template=custom_prompt, input_variables=["text"])
    MAP_PROMPT = PromptTemplate(template="Summarize:\n\n{text}", input_variables=["text"])
    if chain_type == "map_reduce":
        chain = load_summarize_chain(llm, chain_type=chain_type, 
                                    map_prompt=MAP_PROMPT, combine_prompt=COMBINE_PROMPT)
    else:
        chain = load_summarize_chain(llm, chain_type=chain_type)
    summaries = []
    for i in range(num_summaries):
        summary_output = chain({"input_documents": docs}, return_only_outputs=True)["output_text"]
        summaries.append(summary_output)
    
    return summaries

In [ ]:
@st.cache_data
def color_semantic_chunks(docs):
    """
    Tô màu các chunk dựa trên danh sách Document đã được Semantic Chunker xử lý.
    docs: list[Document] từ hàm setup_documents
    """
    # Danh sách màu sắc để phân biệt các đoạn
    chunk_colors = ["#a8d08d", "#c6dbef", "#fdd0a2", "#d9d9d9", "#f9cb9c", "#b4a7d6"]
    
    colored_html = ""
    
    for i, doc in enumerate(docs):
        color = chunk_colors[i % len(chunk_colors)]
        # Lấy nội dung text từ đối tượng Document
        content = doc.page_content
        # Thay thế dấu xuống dòng bằng <br> để hiển thị đúng trên HTML Streamlit
        content = content.replace("\n", "<br>")
        
        # Tạo nhãn để biết đây là Chunk số mấy (tiện cho bạn theo dõi đồ án)
        label = f"<small style='color:black; font-weight:bold;'>[Chunk {i+1}]</small> "
        
        colored_html += f'<mark style="background-color: {color}; padding: 2px; border-radius: 4px; line-height: 2;">{label}{content}</mark> '
    
    return colored_html

2026-03-22 03:04:39.572 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [ ]:
def color_semantic_chunks(docs, max_preview_length=1000, show_metadata=True):
    chunk_colors = ["#e8f5e9", "#fff3e0", "#e3f2fd", "#fce4ec", "#f3e5f5", "#e8f0fe"]
    
    # CSS nên được gọi bên ngoài vòng lặp (giữ nguyên của bạn là rất tốt)
    st.markdown("""
    <style>
    .chunk-container {
        padding: 15px; margin: 10px 0; border-radius: 10px;
        border-left: 5px solid #2c3e50; background-color: white;
        font-family: 'Segoe UI', sans-serif; box-shadow: 2px 2px 5px rgba(0,0,0,0.05);
    }
    .chunk-label {
        font-size: 0.8em; font-weight: bold; color: #555;
        background: rgba(0,0,0,0.05); padding: 3px 10px;
        border-radius: 15px; margin-right: 5px;
    }
    .chunk-content { line-height: 1.6; color: #333; margin-top: 10px; }
    </style>
    """, unsafe_allow_html=True)
    
    html_output = ""
    for i, doc in enumerate(docs):
        color = chunk_colors[i % len(chunk_colors)]
        content = doc.page_content.replace("\n", "<br>") # Thay thế xuống dòng
        
        # Metadata logic
        meta = getattr(doc, 'metadata', {})
        page_info = f"📄 Trang {meta.get('page', 0) + 1}" if 'page' in meta else ""
        
        # Xử lý độ dài hiển thị
        is_long = max_preview_length > 0 and len(doc.page_content) > max_preview_length
        display_text = content[:max_preview_length] + "..." if is_long else content
        
        html_output += f"""
        <div class='chunk-container' style='background-color: {color};'>
            <div>
                <span class='chunk-label'>📌 Đoạn {i+1}</span>
                <span class='chunk-label'>{page_info}</span>
            </div>
            <div class='chunk-content'>{display_text}</div>
        </div>
        """
    return html_output

In [26]:
def main():
    st.set_page_config(layout="wide", page_title="AI Research Paper Summarizer")
    st.title("Custom Summarization App")
    llm_option = st.sidebar.selectbox("LLM Model", ["gpt-3.5-turbo", "gpt-4o"])
    temperature = st.sidebar.slider("Temperature (Độ sáng tạo)", 0.0, 1.0, 0.0, 0.1)
    llm = ChatOpenAI(model_name=llm_option, temperature=temperature)
    # 2. Cấu hình Semantic Chunking
    st.sidebar.subheader("🧠 Semantic Splitting")
    bp_type = st.sidebar.selectbox(
        "Breakpoint Type", 
        ['percentile', 'standard_deviation', 'interquartile'],
        help="Công thức xác định điểm cắt dựa trên độ lệch ngữ nghĩa."
    )
    bp_threshold = st.sidebar.slider(
        "Breakpoint Threshold", 
        min_value=0, max_value=100, value=95,
        help="Ngưỡng để cắt. Percentile nên từ 90-95. Std_dev nên từ 1-3."
    )
    # 3. Cấu hình Summarization
    st.sidebar.subheader("📝 Summary Settings")
    chain_type = st.sidebar.selectbox("Chain Type", ["map_reduce", "stuff", "refine"])
    num_summaries = st.sidebar.number_input("Number of Summaries", 1, 5, 1)

    # --- MAIN CONTENT ---
    tab1, tab2 = st.tabs(["🚀 Summarize Paper", "🔍 Debug & Visualize"])

    with tab1:
        user_prompt = st.text_input("Nhập yêu cầu tóm tắt (Prompt)", "Tóm tắt các luận điểm chính của bài báo này bằng tiếng Việt")
        pdf_file_path = st.text_input("Đường dẫn file (PDF, TXT, JPG, PNG)")

        if pdf_file_path:
            if os.path.exists(pdf_file_path):
                # Xử lý file
                with st.spinner("Đang phân tích ngữ nghĩa và chia đoạn..."):
                    docs = setup_documents(
                        pdf_file_path, 
                        breakpoint_type=bp_type, 
                        breakpoint_threshold=bp_threshold
                    )
                
                if docs:
                    st.success(f"Đã tải xong! Tìm thấy {len(docs)} đoạn nội dung (chunks).")
                    
                    if st.button("🚀 Start Summarizing"):
                        with st.spinner("Đang tóm tắt..."):
                            result = custom_summary(docs, llm, user_prompt, chain_type, num_summaries)
                            st.write("### 📄 Kết quả tóm tắt:")
                            for i, summary in enumerate(result):
                                st.info(f"Bản tóm tắt #{i+1}:\n\n {summary}")
            else:
                st.error("Đường dẫn file không hợp lệ hoặc không tồn tại.")

    with tab2:
        st.header("Visualize Semantic Chunks")
        if 'docs' in locals() and docs:
            st.write("Dưới đây là cách AI 'nhìn' bài báo của bạn sau khi chia theo ngữ nghĩa:")
            # Gọi hàm hiển thị UX xịn của bạn
            html_view = color_semantic_chunks(docs)
            st.markdown(html_view, unsafe_allow_html=True)
        else:
            st.info("Hãy nhập đường dẫn file ở Tab 1 để xem trực quan hóa các đoạn cắt tại đây.")

if __name__ == "__main__":
    main()

2026-03-22 03:14:48.079 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 03:14:48.080 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 03:14:48.081 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 03:14:48.081 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 03:14:48.082 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 03:14:48.084 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 03:14:48.085 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 03:14:48.086 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable